# A CNN-LSTM Hybrid Model for Sign-Language Gesture Recognition
**Domain:** Deep Learning & Computer Vision  
**Keywords:** Sign Language, Gesture Recognition, CNN, BiLSTM, Temporal Attention, Accessibility App  

---
### Abstract
This project develops a CNN-LSTM hybrid model to translate hand gestures into text captions, addressing the problem of sign-language gesture recognition through a data-driven approach. Recorded sign-language video clips form the basis of the dataset, which is processed through a structured pipeline before model building and evaluation. Results are presented through an accessibility communication app, so that the insights generated are directly actionable.

## 1. Environment Setup & Imports

In [ ]:
import os
import cv2
import json
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

## 2. Configuration & Hyperparameters

In [ ]:
CLASSES = ["Hello", "Thank You", "Yes", "No", "Please", "Help", "Sorry", "Love", "Good", "Bad"]
SEQUENCE_LENGTH = 16   # Frames sampled uniformly per clip
FRAME_SIZE = (112, 112) # Height, Width for CNN
BATCH_SIZE = 8
FEATURE_DIM = 256
LSTM_HIDDEN = 128
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4

print(f"Target Vocabulary: {len(CLASSES)} classes")

## 3. Dataset Preprocessing & Uniform Frame Sampling
Videos vary in duration; to pass sequences into a batch tensor $(B, T, C, H, W)$, we apply uniform temporal sub-sampling.

In [ ]:
def sample_video_frames(video_path, seq_len=16, target_size=(112, 112)):
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, max(0, total_frames - 1), seq_len).astype(int)
    
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, target_size)
            frames.append(frame)
        else:
            frames.append(np.zeros((*target_size, 3), dtype=np.uint8))
    cap.release()
    return np.array(frames)

## 4. CNN-LSTM Hybrid Architecture with Temporal Self-Attention
1. **2D-CNN**: Maps each frame into a spatial embedding vector.
2. **BiLSTM**: Models forward and backward gesture trajectory across 16 frames.
3. **Attention Layer**: Computes attention weights $\alpha_t$ to focus on key inflection frames.
4. **Classifier Head**: Maps aggregated representation to gesture classes.

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
    def forward(self, x):
        scores = self.attn(x) # (B, T, 1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(x * weights, dim=1)
        return context, weights.squeeze(-1)

class CustomCNN(nn.Module):
    def __init__(self, feature_dim=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(256, feature_dim), nn.ReLU(True), nn.Dropout(0.3))
    def forward(self, x):
        return self.fc(self.features(x))

class CNNLSTMModel(nn.Module):
    def __init__(self, num_classes, feature_dim=256, hidden_dim=128):
        super().__init__()
        self.cnn = CustomCNN(feature_dim)
        self.lstm = nn.LSTM(feature_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.attention = TemporalAttention(hidden_dim * 2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.LayerNorm(64),
            nn.ReLU(True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        B, T, C, H, W = x.size()
        cnn_feats = self.cnn(x.view(B * T, C, H, W))
        lstm_out, _ = self.lstm(cnn_feats.view(B, T, -1))
        context, attn_weights = self.attention(lstm_out)
        logits = self.classifier(context)
        return logits, attn_weights

model = CNNLSTMModel(num_classes=len(CLASSES)).to(device)
print(f"Model instantiated with {sum(p.numel() for p in model.parameters()):,} parameters.")

## 5. Model Verification with Synthetic Forward Pass

In [ ]:
# Test tensor: 2 video batches of 16 frames each (112x112 RGB)
sample_video = torch.randn(2, 16, 3, 112, 112).to(device)
with torch.no_grad():
    logits, weights = model(sample_video)
    probs = torch.softmax(logits, dim=1)

print(f"Logits shape: {logits.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Predicted classes: {torch.argmax(probs, dim=1).cpu().numpy()}")

## 6. Real-Time Accessibility Communication App
The model is deployed via Streamlit in `app.py`. To run the interactive web interface:
```bash
streamlit run app.py
```
This opens a local UI providing:
- Live video/webcam gesture translation
- High-contrast visual caption banner
- Text-to-Speech (TTS) voice synthesis
- Continuous sentence constructor